# 04 — BERT Embedding Generation

This notebook generates 768-dim SecBERT embeddings for all CVE descriptions.

**Run order:**
1. Cell 1 — Mount Drive + set paths
2. Cell 2 — Install libraries + load SecBERT model
3. Cell 3 — Define embedding function
4. Cell 4 — Generate embeddings (smart: handles first run AND update runs)
5. Cell 5 — Verify saved file

---
**First time running:** generates all embeddings from scratch (~35 min)

**After live updater:** only generates embeddings for NEW rows (~15-30 min)

## Cell 1 — Mount Drive + set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE       = '/content/drive/MyDrive/CVE_Project'
PROCESSED  = f'{BASE}/processed'
EMBEDDINGS = f'{BASE}/embeddings'
os.makedirs(EMBEDDINGS, exist_ok=True)

# Load processed CSV
df = pd.read_csv(f'{PROCESSED}/cves_processed.csv')
print(f'Processed CSV loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')

Mounted at /content/drive
Processed CSV loaded: 200431 rows
Columns: ['cve_id', 'description', 'cvss_score', 'cvss_label', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope', 'description_clean', 'entity_count', 'entities', 'has_remote', 'has_unauth', 'has_exec', 'has_priv_esc', 'has_dos', 'has_overflow', 'desc_word_count', 'attack_vector_enc', 'attack_complexity_enc', 'privileges_required_enc', 'user_interaction_enc', 'scope_enc']


## Cell 2 — Install libraries + load SecBERT

In [2]:
!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModel
import torch

# Check GPU
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'jackaduma/SecBERT'
print(f'\nLoading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = model.to(device)
model.eval()

print(f'Model loaded on: {device}')

GPU available: True
GPU name: Tesla T4

Loading jackaduma/SecBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda


## Cell 3 — Define embedding function

In [3]:
import numpy as np

def get_embeddings_batch(texts, batch_size=64):
    """
    Generate 768-dim CLS token embeddings for a list of texts.
    Processes in batches for memory efficiency.
    """
    all_embeddings = []
    total          = len(texts)

    for i in range(0, total, batch_size):
        batch  = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors = 'pt',
            truncation     = True,
            max_length     = 512,
            padding        = True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token = index 0 of last hidden state
        batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(batch_emb)

        # Progress every 10 batches
        if (i // batch_size) % 10 == 0:
            done = min(i + batch_size, total)
            print(f'  {done}/{total} ({100*done//total}%)')

    return np.vstack(all_embeddings)

print('Embedding function ready.')

Embedding function ready.


## Cell 4 — Generate embeddings

**This cell is smart — it automatically handles both cases:**
- If no embeddings file exists → generates all from scratch
- If embeddings file exists → only generates embeddings for NEW rows and appends

In [4]:
import os

EMB_FILE  = f'{EMBEDDINGS}/bert_embeddings.npy'
TEMP_FILE = f'{EMBEDDINGS}/bert_embeddings_temp.npy'

total_rows = len(df)

if os.path.exists(EMB_FILE):
    # ── UPDATE MODE: only embed new rows ──────────────────────────────────
    old_emb   = np.load(EMB_FILE)
    old_count = len(old_emb)
    new_texts = df['description_clean'].iloc[old_count:].tolist()

    print(f'Mode:              UPDATE (append only)')
    print(f'Existing embeddings: {old_count}')
    print(f'Total rows in CSV:   {total_rows}')
    print(f'New rows to embed:   {len(new_texts)}')

    if len(new_texts) == 0:
        print('\nNo new rows to embed. Everything is up to date.')
        combined = old_emb
    else:
        print(f'\nGenerating {len(new_texts)} new embeddings...')
        new_emb  = get_embeddings_batch(new_texts, batch_size=64)
        combined = np.vstack([old_emb, new_emb])
        print(f'\nCombined shape: {combined.shape}')

        # Save to temp first, verify, then overwrite original
        print('Saving to temp file...')
        np.save(TEMP_FILE, combined)
        verify = np.load(TEMP_FILE)

        if verify.shape == (total_rows, 768):
            np.save(EMB_FILE, combined)
            os.remove(TEMP_FILE)
            print(f'Saved successfully: {verify.shape}')
        else:
            print(f'ERROR: temp file shape {verify.shape} does not match expected ({total_rows}, 768)')
            print('Original file NOT overwritten. Check and retry.')

else:
    # ── FIRST RUN: generate all from scratch ──────────────────────────────
    texts = df['description_clean'].tolist()

    print(f'Mode:              FIRST RUN (generate all)')
    print(f'Total rows to embed: {len(texts)}')
    print(f'Estimated time:    30-50 minutes on T4 GPU')
    print(f'Do not close this tab.\n')

    combined = get_embeddings_batch(texts, batch_size=64)
    print(f'\nGenerated shape: {combined.shape}')

    # Save to temp first, verify, then save as final
    print('Saving to temp file...')
    np.save(TEMP_FILE, combined)
    verify = np.load(TEMP_FILE)

    if verify.shape == (total_rows, 768):
        np.save(EMB_FILE, combined)
        os.remove(TEMP_FILE)
        print(f'Saved successfully: {verify.shape}')
    else:
        print(f'ERROR: temp file shape {verify.shape} does not match expected ({total_rows}, 768)')
        print('File NOT saved. Check and retry.')

Mode:              UPDATE (append only)
Existing embeddings: 200431
Total rows in CSV:   200431
New rows to embed:   0

No new rows to embed. Everything is up to date.


## Cell 5 — Final verification

In [5]:
import numpy as np

reloaded = np.load(f'{EMBEDDINGS}/bert_embeddings.npy')
csv_rows = len(pd.read_csv(f'{PROCESSED}/cves_processed.csv'))

print(f'Embedding shape:   {reloaded.shape}')
print(f'CSV rows:          {csv_rows}')
print(f'Rows match:        {reloaded.shape[0] == csv_rows}')
print(f'Embedding dims:    {reloaded.shape[1]} (should be 768)')
print(f'Dims correct:      {reloaded.shape[1] == 768}')
print()
if reloaded.shape[0] == csv_rows and reloaded.shape[1] == 768:
    print('All checks passed. Ready to run 05_training.ipynb')
else:
    print('MISMATCH — do not proceed to training. Re-run Cell 4.')

Embedding shape:   (200431, 768)
CSV rows:          200431
Rows match:        True
Embedding dims:    768 (should be 768)
Dims correct:      True

All checks passed. Ready to run 05_training.ipynb
